### 1 - Validando de arquivos para o laboratório.


In [1]:
from pathlib import Path

pcap_path = Path("data/worm/extraido/pacotes_worm.pcapng")

print("Arquivo encontrado:", pcap_path.exists())

if pcap_path.exists():
    print(f"Tamanho: {pcap_path.stat().st_size:,} bytes")
    print(f"Caminho absoluto: {pcap_path.resolve()}")
else:
    print("Arquivo não encontrado.")

Arquivo encontrado: True
Tamanho: 144,365,932 bytes
Caminho absoluto: C:\Users\marce\GitHub\network-anomaly-detection-lab\data\worm\extraido\pacotes_worm.pcapng


Resultado da validação
- Arquivo encontrado: True
- Tamanho: 144,365,932 bytes

---

### 2 - Validando arquivo da captura .pcap

O Scapy oferece RawPcapNgReader, que permite percorrer a captura pacote a pacote, evitando manter todos os objetos de pacote na memória simultaneamente.

In [2]:
from scapy.utils import RawPcapNgReader

total_pacotes = 0

with RawPcapNgReader(str(pcap_path)) as leitor:
    for pacote, metadados in leitor:
        total_pacotes += 1

print(f"Total de pacotes na captura: {total_pacotes:,}")

Total de pacotes na captura: 206,536


Resultado da captura: Total de pacotes na captura: 206,536

---

### 3 - Contador pacotes TCP e tráfego nas portas 443 e 445
Como interpretar os resultados:
- TCP: quantidade de pacotes que contêm o protocolo TCP.
- Porta 443: normalmente associada ao HTTPS; a contagem inclui TCP e UDP.
- Porta 445: normalmente associada ao SMB, relevante para investigar possíveis tentativas de propagação do worm.


In [3]:
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP, UDP
from scapy.layers.inet6 import IPv6

total_pacotes = 0
total_tcp = 0
porta_443 = 0
porta_445 = 0

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        total_pacotes += 1

        if TCP in pkt:
            total_tcp += 1

        # Contabiliza pacotes TCP ou UDP envolvendo cada porta.
        if TCP in pkt:
            porta_origem = pkt[TCP].sport
            porta_destino = pkt[TCP].dport
        elif UDP in pkt:
            porta_origem = pkt[UDP].sport
            porta_destino = pkt[UDP].dport
        else:
            continue

        if 443 in (porta_origem, porta_destino):
            porta_443 += 1

        if 445 in (porta_origem, porta_destino):
            porta_445 += 1

print(f"Total de pacotes: {total_pacotes:,}")
print(f"Pacotes TCP: {total_tcp:,}")
print(f"Pacotes envolvendo a porta 443 (TCP/UDP): {porta_443:,}")
print(f"Pacotes envolvendo a porta 445 (TCP/UDP): {porta_445:,}")

Total de pacotes: 206,536
Pacotes TCP: 182,823
Pacotes envolvendo a porta 443 (TCP/UDP): 19,456
Pacotes envolvendo a porta 445 (TCP/UDP): 2,014


Resultado da contagem

- Total de pacotes: 206,536
- Pacotes TCP: 182,823
- Pacotes envolvendo a porta 443 (TCP/UDP): 19,456
- Pacotes envolvendo a porta 445 (TCP/UDP): 2,014

---

### 4 - Descobrir IPs de origem utilizando a porta 445
Agora vamos identificar os endereços IP de origem que mais enviam pacotes envolvendo a porta 445. Também registraremos quantos destinos distintos cada origem atinge.

O que vamos aprender com esse resultado?
- Muitos pacotes na porta 445: pode indicar atividade SMB intensa.
- Muitos destinos distintos: pode ser compatível com varredura ou tentativa de propagação.
- Poucos destinos: pode representar comunicação normal entre servidores ou compartilhamentos.


In [4]:

from collections import Counter, defaultdict
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP, UDP

pacotes_por_origem = Counter()
destinos_445_por_origem = defaultdict(set)
pacotes_445_por_origem = Counter()

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt:
            continue

        ip_origem = pkt[IP].src
        ip_destino = pkt[IP].dst

        pacotes_por_origem[ip_origem] += 1

        if TCP in pkt:
            porta_origem = pkt[TCP].sport
            porta_destino = pkt[TCP].dport
        elif UDP in pkt:
            porta_origem = pkt[UDP].sport
            porta_destino = pkt[UDP].dport
        else:
            continue

        if 445 in (porta_origem, porta_destino):
            pacotes_445_por_origem[ip_origem] += 1
            destinos_445_por_origem[ip_origem].add(ip_destino)

print("Top 10 origens por pacotes envolvendo a porta 445:")
for ip, quantidade in pacotes_445_por_origem.most_common(10):
    print(
        f"{ip:>15} | "
        f"pacotes na porta 445: {quantidade:>6} | "
        f"destinos distintos: {len(destinos_445_por_origem[ip]):>4}"
    )


Top 10 origens por pacotes envolvendo a porta 445:
  172.19.15.200 | pacotes na porta 445:   1920 | destinos distintos:  269
    172.19.15.4 | pacotes na porta 445:     14 | destinos distintos:    1
    172.19.15.5 | pacotes na porta 445:     14 | destinos distintos:    1
    172.19.15.8 | pacotes na porta 445:     14 | destinos distintos:    1
   172.19.15.11 | pacotes na porta 445:     14 | destinos distintos:    1
   172.19.15.10 | pacotes na porta 445:     14 | destinos distintos:    1
    172.19.15.9 | pacotes na porta 445:     14 | destinos distintos:    1
   172.19.15.12 | pacotes na porta 445:     10 | destinos distintos:    1


Resultado

### Top origens por pacotes envolvendo a porta 445

| IP de origem | Pacotes na porta 445 | Destinos distintos |
|---|---:|---:|
| 172.19.15.200 | 1.920 | 269 |
| 172.19.15.4 | 14 | 1 |
| 172.19.15.5 | 14 | 1 |
| 172.19.15.8 | 14 | 1 |
| 172.19.15.11 | 14 | 1 |
| 172.19.15.10 | 14 | 1 |
| 172.19.15.9 | 14 | 1 |
| 172.19.15.12 | 10 | 1 |


- IP com maior atividade envolvendo a porta 445 --> 172.19.15.200
   
- Pacotes envolvendo a porta 445 --> 1.920

- Destinos distintos contabilizados --> 269

- Esse IP responde por aproximadamente 95,3% dos 2.014 pacotes contabilizados na porta 445.

A diferença é expressiva: o IP 172.19.15.200 aparece com 1.920 pacotes, enquanto os demais endereços apresentados aparecem com apenas 10 a 14 pacotes cada.

Isso é compatível com um host realizando varredura ou tentando propagar uma infecção via SMB

---

### 5 - Separar tentativas de conexão e respostas 

Vamos examinar especificamente os pacotes TCP destinados à porta 445, distinguindo tentativas de conexão (SYN) e respostas. Isso ajudará a entender se o host está iniciando conexões contra muitos destinos.

O que esse teste nos dirá?

- Muitos SYN enviados a partir de 172.19.15.200 para destinos diferentes reforçariam a hipótese de varredura.
- Respostas SYN-ACK indicariam destinos que responderam positivamente à abertura da conexão.
- Pacotes RST podem indicar conexões recusadas ou encerradas, mas sua interpretação depende do contexto.

In [5]:

from collections import Counter
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

syn_por_destino = Counter()
syn_ack_por_origem = Counter()
rst_por_origem = Counter()
portas_origem = Counter()

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        ip_src = pkt[IP].src
        ip_dst = pkt[IP].dst
        tcp = pkt[TCP]

        # Tentativas de conexão para a porta 445 (SYN sem ACK)
        if tcp.dport == 445 and tcp.flags & 0x02 and not tcp.flags & 0x10:
            syn_por_destino[ip_src] += 1
            portas_origem[ip_src] += 1

        # Respostas SYN-ACK vindas da porta 445
        if tcp.sport == 445 and tcp.flags & 0x12 == 0x12:
            syn_ack_por_origem[ip_src] += 1

        # Pacotes RST originados por cada IP
        if tcp.flags & 0x04:
            rst_por_origem[ip_src] += 1

print("Top 10 IPs por tentativas SYN para a porta 445:")
for ip, qtd in syn_por_destino.most_common(10):
    print(f"{ip:>15} | SYN enviados para destino: {qtd}")

print("\nRespostas SYN-ACK originadas pela porta 445:")
for ip, qtd in syn_ack_por_origem.most_common(10):
    print(f"{ip:>15} | SYN-ACK: {qtd}")

print("\nTop 10 IPs por pacotes RST:")
for ip, qtd in rst_por_origem.most_common(10):
    print(f"{ip:>15} | RST: {qtd}")


Top 10 IPs por tentativas SYN para a porta 445:
  172.19.15.200 | SYN enviados para destino: 1912

Respostas SYN-ACK originadas pela porta 445:
    172.19.15.4 | SYN-ACK: 14
    172.19.15.5 | SYN-ACK: 14
    172.19.15.8 | SYN-ACK: 14
   172.19.15.11 | SYN-ACK: 14
   172.19.15.10 | SYN-ACK: 14
    172.19.15.9 | SYN-ACK: 14
   172.19.15.12 | SYN-ACK: 10

Top 10 IPs por pacotes RST:
    172.19.15.6 | RST: 35
    172.19.15.7 | RST: 8
  172.19.15.200 | RST: 4
  52.123.128.14 | RST: 4
   72.153.5.134 | RST: 4
    172.19.15.2 | RST: 2
    40.79.167.8 | RST: 1
   4.160.58.170 | RST: 1
    72.153.5.63 | RST: 1
   72.153.5.129 | RST: 1


Resultado

### Tentativas SYN para a porta 445

| IP de origem | SYNs enviados |
|---|---:|
| 172.19.15.200 | 1.912 |

### Respostas SYN-ACK originadas pela porta 445

| IP de origem | SYN-ACKs |
|---|---:|
| 172.19.15.4 | 14 |
| 172.19.15.5 | 14 |
| 172.19.15.8 | 14 |
| 172.19.15.11 | 14 |
| 172.19.15.10 | 14 |
| 172.19.15.9 | 14 |
| 172.19.15.12 | 10 |

### Top 10 IPs por pacotes RST

| IP | Pacotes RST |
|---|---:|
| 172.19.15.6 | 35 |
| 172.19.15.7 | 8 |
| 172.19.15.200 | 4 |
| 52.123.128.14 | 4 |
| 72.153.5.134 | 4 |
| 172.19.15.2 | 2 |
| 40.79.167.8 | 1 |
| 4.160.58.170 | 1 |
| 72.153.5.63 | 1 |
| 72.153.5.129 | 1 |

O resultado reforça a hipótese de que o IP 172.19.15.200 está associado à atividade de propagação ou varredura pela porta SMB (TCP/445)

O que os resultados mostram:

IP com mais tentativas SYN para TCP/445 sendo o 172.19.15.200 com 1.912 pacotes SYN identificados;

Múltiplos destinos SMB,  269 endereços de destino distintos associados ao tráfego envolvendo a porta 445;

Respostas de outros hosts, Os IPs de .4, .5, .8, .9, .10, .11 e .12 enviaram respostas SYN-ACK a partir da porta 445.



---

### 6 - Descobrir hosts que receberam SYNs

- Precisamos confirmar se os 1.912 SYNs foram distribuídos entre muitos destinos e descobrir quando a atividade começou. Isso ajuda a distinguir uma varredura de conexões repetidas com poucos hosts e prepara a investigação temporal do possível patient zero.

- O código calcula três informações independentes: quantidade de SYNs, número de destinos únicos e horário do primeiro SYN por IP. O horário será apresentado em UTC.

In [6]:

from collections import Counter, defaultdict
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

syn_por_origem = Counter()
destinos_por_origem = defaultdict(set)
primeiro_syn_por_origem = {}

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        ip_src = pkt[IP].src
        ip_dst = pkt[IP].dst
        tcp = pkt[TCP]

        # SYN inicial destinado à porta SMB (445)
        if tcp.dport == 445 and (int(tcp.flags) & 0x02) and not (int(tcp.flags) & 0x10):
            syn_por_origem[ip_src] += 1
            destinos_por_origem[ip_src].add(ip_dst)

            timestamp = float(pkt.time)
            if (
                ip_src not in primeiro_syn_por_origem
                or timestamp < primeiro_syn_por_origem[ip_src]
            ):
                primeiro_syn_por_origem[ip_src] = timestamp

print("Origens de SYN para TCP/445:")
print(f"{'IP de origem':<18} {'SYNs':>8} {'Destinos únicos':>17} {'Primeiro SYN':>24}")

for ip, qtd in syn_por_origem.most_common(10):
    primeiro = primeiro_syn_por_origem[ip]
    from datetime import datetime, timezone

    horario = datetime.fromtimestamp(
        primeiro, tz=timezone.utc
    ).strftime("%Y-%m-%d %H:%M:%S UTC")

    print(f"{ip:<18} {qtd:>8} {len(destinos_por_origem[ip]):>17} {horario:>24}")


Origens de SYN para TCP/445:
IP de origem           SYNs   Destinos únicos             Primeiro SYN
172.19.15.200          1912               269  2025-11-02 19:33:31 UTC


Resultado 

Origens de SYN para TCP/445:

IP de origem           SYNs   Destinos únicos             Primeiro SYN
172.19.15.200          1912               269  2025-11-02 19:33:31 UTC

- A média foi de aproximadamente 7,1 SYNs por destino único. Esse comportamento é compatível com tentativas repetidas de conexão SMB distribuídas por muitos hosts.

- Um detalhe importante: o primeiro SYN registrado é o primeiro observado na captura, não necessariamente o primeiro evento ocorrido na rede. Portanto, ainda não podemos afirmar que esse host foi o patient zero.

---

### 7 - Sequência temporal

Vamos verificar se o tráfego SMB começou nesse horário ou se o host já vinha se comunicando com outros equipamentos antes disso. Essa análise pode ajudar a identificar padrões de propagação.

O que vamos procurar
- Se 172.19.15.200 aparece muito antes dos demais hosts ou se vários endereços já apresentam atividade SMB antes dele.
- Se os outros hosts começam a responder em sequência após as tentativas de conexão.
- Se existem outros IPs com atividade SMB anterior, que possam representar uma origem alternativa ou uma etapa anterior da propagação.

In [7]:

from collections import defaultdict
from datetime import datetime, timezone
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

# Para cada IP, registrar o primeiro e o último pacote SMB
tempos_smb = defaultdict(lambda: {"primeiro": None, "ultimo": None})
contagem_smb = defaultdict(int)

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]

        # Considera tráfego TCP envolvendo a porta 445
        if tcp.sport != 445 and tcp.dport != 445:
            continue

        ip_src = pkt[IP].src
        ts = float(pkt.time)

        contagem_smb[ip_src] += 1

        if tempos_smb[ip_src]["primeiro"] is None or ts < tempos_smb[ip_src]["primeiro"]:
            tempos_smb[ip_src]["primeiro"] = ts

        if tempos_smb[ip_src]["ultimo"] is None or ts > tempos_smb[ip_src]["ultimo"]:
            tempos_smb[ip_src]["ultimo"] = ts

print(f"{'IP':<18} {'Pacotes SMB':>12} {'Primeiro evento (UTC)':>25} {'Último evento (UTC)':>25}")

for ip, qtd in sorted(
    contagem_smb.items(),
    key=lambda item: tempos_smb[item[0]]["primeiro"]
):
    primeiro = datetime.fromtimestamp(
        tempos_smb[ip]["primeiro"], tz=timezone.utc
    ).strftime("%Y-%m-%d %H:%M:%S")
    ultimo = datetime.fromtimestamp(
        tempos_smb[ip]["ultimo"], tz=timezone.utc
    ).strftime("%Y-%m-%d %H:%M:%S")

    print(f"{ip:<18} {qtd:>12} {primeiro:>25} {ultimo:>25}")


IP                  Pacotes SMB     Primeiro evento (UTC)       Último evento (UTC)
172.19.15.200              1920       2025-11-02 19:33:31       2025-11-02 20:41:24
172.19.15.4                  14       2025-11-02 19:33:31       2025-11-02 19:33:37
172.19.15.12                 10       2025-11-02 19:33:31       2025-11-02 19:33:34
172.19.15.5                  14       2025-11-02 19:33:31       2025-11-02 19:33:37
172.19.15.8                  14       2025-11-02 19:33:31       2025-11-02 19:33:37
172.19.15.11                 14       2025-11-02 19:33:31       2025-11-02 19:33:37
172.19.15.10                 14       2025-11-02 19:33:31       2025-11-02 19:33:37
172.19.15.9                  14       2025-11-02 19:33:31       2025-11-02 19:33:37


Resultado

### Pacotes SMB por IP

| IP | Pacotes SMB | Primeiro evento (UTC) | Último evento (UTC) |
|---|---:|---|---|
| 172.19.15.200 | 1.920 | 2025-11-02 19:33:31 | 2025-11-02 20:41:24 |
| 172.19.15.4 | 14 | 2025-11-02 19:33:31 | 2025-11-02 19:33:37 |
| 172.19.15.12 | 10 | 2025-11-02 19:33:31 | 2025-11-02 19:33:34 |
| 172.19.15.5 | 14 | 2025-11-02 19:33:31 | 2025-11-02 19:33:37 |
| 172.19.15.8 | 14 | 2025-11-02 19:33:31 | 2025-11-02 19:33:37 |
| 172.19.15.11 | 14 | 2025-11-02 19:33:31 | 2025-11-02 19:33:37 |
| 172.19.15.10 | 14 | 2025-11-02 19:33:31 | 2025-11-02 19:33:37 |
| 172.19.15.9 | 14 | 2025-11-02 19:33:31 | 2025-11-02 19:33:37 |

Interpretação da linha do tempo

19:33:31 UTC — início observado
* O primeiro evento SMB registrado para todos os IPs listados ocorre no mesmo segundo.

172.19.15.200 — atividade prolongada
* 1.920 pacotes
* De 19:33:31 a 20:41:24 UTC, com SYNs destinados a 269 endereços distintos.

Outros sete hosts
* Os pacotes SMB registrados para eles se concentram entre 19:33:31 e 19:33:37 UTC.

---

### 8 - Reconstruir primeiras conexões

Vamos examinar os primeiros 30 segundos de tráfego envolvendo a porta 445, preservando horário, origem, destino e flags TCP. Assim poderemos verificar a sequência de tentativas e respostas.

In [8]:

from datetime import datetime, timezone
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

inicio = datetime(2025, 11, 2, 19, 33, 31, tzinfo=timezone.utc).timestamp()
fim = inicio + 30

eventos = []

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]

        if tcp.sport != 445 and tcp.dport != 445:
            continue

        ts = float(pkt.time)

        if inicio <= ts <= fim:
            flags = str(tcp.flags)

            if tcp.dport == 445 and (int(tcp.flags) & 0x02) and not (int(tcp.flags) & 0x10):
                tipo = "SYN -> 445"
            elif tcp.sport == 445 and (int(tcp.flags) & 0x12) == 0x12:
                tipo = "SYN-ACK <- 445"
            elif int(tcp.flags) & 0x04:
                tipo = "RST"
            else:
                tipo = "OUTRO"

            eventos.append((
                ts, pkt[IP].src, pkt[IP].dst,
                tcp.sport, tcp.dport, flags, tipo
            ))

eventos.sort(key=lambda x: x[0])

print(f"Eventos encontrados nos primeiros 30 segundos: {len(eventos)}")
print(f"{'Horário UTC':<12} {'Origem':<16} {'Destino':<16} "
      f"{'Portas':<13} {'Flags':<7} {'Tipo'}")

for ts, src, dst, sport, dport, flags, tipo in eventos[:60]:
    horario = datetime.fromtimestamp(
        ts, tz=timezone.utc
    ).strftime("%H:%M:%S.%f")[:-3]

    portas = f"{sport}->{dport}"
    print(f"{horario:<12} {src:<16} {dst:<16} "
          f"{portas:<13} {flags:<7} {tipo}")


Eventos encontrados nos primeiros 30 segundos: 212
Horário UTC  Origem           Destino          Portas        Flags   Tipo
19:33:31.212 172.19.15.200    172.19.15.2      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.4      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.5      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.6      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.7      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.8      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.9      48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.10     48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.11     48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.12     48903->445    S       SYN -> 445
19:33:31.212 172.19.15.200    172.19.15.2      48903->445    S       SYN ->

Resultado

### Eventos encontrados nos primeiros 30 segundos

**Total de eventos:** 212

| Horário UTC | Origem | Destino | Portas | Flags | Tipo |
|---|---|---|---|---|---|
| 19:33:31.212 | 172.19.15.200 | 172.19.15.2 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.4 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.5 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.6 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.7 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.8 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.9 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.10 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.11 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.12 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.2 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.4 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.5 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.6 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.7 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.8 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.9 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.10 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.11 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.212 | 172.19.15.200 | 172.19.15.12 | 48903 → 445 | S | SYN → 445 |
| 19:33:31.213 | 172.19.15.4 | 172.19.15.200 | 445 → 48903 | SA | SYN-ACK ← 445 |
| 19:33:31.213 | 172.19.15.12 | 172.19.15.200 | 445 → 48903 | SA | SYN-ACK ← 445 |
| 19:33:31.213 | 172.19.15.5 | 172.19.15.200 | 445 → 48903 | SA | SYN-ACK ← 445 |
| 19:33:31.545 | 172.19.15.200 | 172.19.15.10 | 38216 → 445 | S | SYN → 445 |
| 19:33:31.545 | 172.19.15.200 | 172.19.15.11 | 55918 → 445 | S | SYN → 445 |
| 19:33:31.545 | 172.19.15.200 | 172.19.15.12 | 36172 → 445 | S | SYN → 445 |
| 19:33:31.546 | 172.19.15.200 | 172.19.15.12 | 36172 → 445 | S | SYN → 445 |



O que a captura demonstra

- Varredura SMB confirmada como padrão de tráfego [Às 19:33:31.212 UTC, o host .200 envia SYNs para diversos endereços, incluindo .2 e .4 a .12.]

- Há respostas de conexão[Já às 19:33:31.213 UTC, vemos SYN-ACKs de .4, .5 e .12 para .200, indicando resposta ao início das conexões.]

- Possíveis retransmissões [Os mesmos SYNs aparecem repetidos no mesmo milissegundo, usando a porta de origem 48903. Precisamos distinguir duplicação real na captura de retransmissões ou outros efeitos da coleta.]

---

### 9 - Cruzamento de informações

Vamos contar as combinações únicas de origem, destino, porta de origem, porta de destino e número de sequência TCP. Isso nos ajudará a identificar se os SYNs repetidos são pacotes idênticos ou novas tentativas.

Por que isso importa?
Se encontrarmos muitos SYNs totais, mas bem menos SYNs únicos, parte do volume pode ser explicada por pacotes repetidos. Se os números forem próximos, teremos evidência de que o volume representa tentativas distintas de conexão.

In [9]:

from collections import Counter
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

syn_total = Counter()
syn_unicos = Counter()
chaves_syn = set()
duplicatas = Counter()
destinos_unicos = {}

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        ip = pkt[IP]
        tcp = pkt[TCP]

        # SYN inicial para a porta 445, sem ACK
        if tcp.dport == 445 and (int(tcp.flags) & 0x02) and not (int(tcp.flags) & 0x10):
            src = ip.src
            dst = ip.dst

            syn_total[src] += 1

            # Identifica o SYN pela tupla TCP e número de sequência
            chave = (src, dst, tcp.sport, tcp.dport, tcp.seq)

            if chave not in chaves_syn:
                chaves_syn.add(chave)
                syn_unicos[src] += 1

            destinos_unicos.setdefault(src, set()).add(dst)

print(f"{'IP de origem':<18} {'SYNs totais':>12} "
      f"{'SYNs únicos':>12} {'Destinos únicos':>16}")

for ip, total in syn_total.most_common(10):
    print(f"{ip:<18} {total:>12} {syn_unicos[ip]:>12} "
          f"{len(destinos_unicos[ip]):>16}")

print("\nExemplos de SYNs repetidos por origem/destino/portas/sequência:")
frequencias = Counter()

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]
        if tcp.dport == 445 and (int(tcp.flags) & 0x02) and not (int(tcp.flags) & 0x10):
            chave = (
                pkt[IP].src, pkt[IP].dst,
                tcp.sport, tcp.dport, tcp.seq
            )
            frequencias[chave] += 1

exemplos = [(chave, qtd) for chave, qtd in frequencias.items() if qtd > 1]

for chave, qtd in sorted(exemplos, key=lambda item: item[1], reverse=True)[:10]:
    src, dst, sport, dport, seq = chave
    print(f"{src}:{sport} -> {dst}:{dport} | seq={seq} | ocorrências={qtd}")


IP de origem        SYNs totais  SYNs únicos  Destinos únicos
172.19.15.200              1912          289              269

Exemplos de SYNs repetidos por origem/destino/portas/sequência:
172.19.15.200:42498 -> 172.19.15.4:445 | seq=2333550445 | ocorrências=10
172.19.15.200:60180 -> 172.19.15.5:445 | seq=3723666989 | ocorrências=10
172.19.15.200:58498 -> 172.19.15.8:445 | seq=4082807981 | ocorrências=10
172.19.15.200:42166 -> 172.19.15.9:445 | seq=1480442110 | ocorrências=10
172.19.15.200:38216 -> 172.19.15.10:445 | seq=909950930 | ocorrências=10
172.19.15.200:55918 -> 172.19.15.11:445 | seq=3671497859 | ocorrências=10
172.19.15.200:36172 -> 172.19.15.12:445 | seq=3861954819 | ocorrências=8
172.19.15.200:43369 -> 172.15.19.0:445 | seq=1634816967 | ocorrências=7
172.19.15.200:33233 -> 172.15.19.1:445 | seq=3841037866 | ocorrências=7
172.19.15.200:45671 -> 172.15.19.2:445 | seq=3193496911 | ocorrências=7


Resultado 

### Resumo dos SYNs por origem

| IP de origem | SYNs totais | SYNs únicos | Destinos únicos |
|---|---:|---:|---:|
| 172.19.15.200 | 1.912 | 289 | 269 |

### Exemplos de SYNs repetidos

| Origem:porta | Destino:porta | Sequência | Ocorrências |
|---|---|---:|---:|
| 172.19.15.200:42498 | 172.19.15.4:445 | 2333550445 | 10 |
| 172.19.15.200:60180 | 172.19.15.5:445 | 3723666989 | 10 |
| 172.19.15.200:58498 | 172.19.15.8:445 | 4082807981 | 10 |
| 172.19.15.200:42166 | 172.19.15.9:445 | 1480442110 | 10 |
| 172.19.15.200:38216 | 172.19.15.10:445 | 909950930 | 10 |
| 172.19.15.200:55918 | 172.19.15.11:445 | 3671497859 | 10 |
| 172.19.15.200:36172 | 172.19.15.12:445 | 3861954819 | 8 |
| 172.19.15.200:43369 | 172.15.19.0:445 | 1634816967 | 7 |
| 172.19.15.200:33233 | 172.15.19.1:445 | 3841037866 | 7 |
| 172.19.15.200:45671 | 172.15.19.2:445 | 3193496911 | 7 |

Análise

- SYNs capturados --> 1.912
- Destinos únicos --> 269
- Tuplas de SYN distintas --> 289
- Pacotes além do primeiro SYN de cada tupla --> 84,9%
- Principal origem de tráfego SMB suspeito --> 172.19.15.200

A porcentagem acima é calculada como \((1912-289)/1912\), ou aproximadamente 84,9%. Ela representa a proporção de pacotes que excedem a primeira ocorrência de cada tupla identificada — não significa necessariamente que todos sejam retransmissões causadas por perda de pacotes.

---

### 10 - Distribuição dos Destinos

Vamos descobrir se a atividade se concentra em uma sub-rede ou se percorre diversos intervalos de IP. Isso pode ajudar a caracterizar o padrão de varredura.

In [10]:

from collections import Counter, defaultdict
from ipaddress import ip_address, ip_network
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

destinos_por_origem = defaultdict(set)
syn_por_subrede = Counter()
syn_por_destino = Counter()

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]

        if tcp.dport != 445:
            continue

        flags = int(tcp.flags)
        if not (flags & 0x02) or (flags & 0x10):
            continue

        origem = pkt[IP].src
        destino = pkt[IP].dst

        destinos_por_origem[origem].add(destino)
        syn_por_destino[destino] += 1

        # Agrupa o destino pela rede IPv4 /24
        rede = str(ip_network(f"{destino}/24", strict=False))
        syn_por_subrede[rede] += 1

origem = "172.19.15.200"

print(f"Destinos únicos de {origem}: {len(destinos_por_origem[origem])}")

print("\nTop 15 destinos por quantidade de SYNs:")
for ip, qtd in syn_por_destino.most_common(15):
    print(f"{ip:>15} | {qtd:>4} SYNs")

print("\nDistribuição por sub-rede /24:")
for rede, qtd in syn_por_subrede.most_common(20):
    print(f"{rede:>18} | {qtd:>5} SYNs")


Destinos únicos de 172.19.15.200: 269

Top 15 destinos por quantidade de SYNs:
    172.19.15.4 |   14 SYNs
    172.19.15.5 |   14 SYNs
    172.19.15.8 |   14 SYNs
    172.19.15.9 |   14 SYNs
   172.19.15.10 |   14 SYNs
   172.19.15.11 |   14 SYNs
    172.19.15.2 |   10 SYNs
   172.19.15.12 |   10 SYNs
    172.15.19.0 |    7 SYNs
    172.15.19.1 |    7 SYNs
    172.15.19.2 |    7 SYNs
    172.15.19.3 |    7 SYNs
    172.15.19.4 |    7 SYNs
    172.15.19.5 |    7 SYNs
    172.15.19.6 |    7 SYNs

Distribuição por sub-rede /24:
    172.15.19.0/24 |  1792 SYNs
    172.19.15.0/24 |   120 SYNs


Resultado

### Resumo

| Origem | Destinos únicos |
|---|---:|
| 172.19.15.200 | 269 |

### Top 15 destinos por quantidade de SYNs

| Destino | SYNs |
|---|---:|
| 172.19.15.4 | 14 |
| 172.19.15.5 | 14 |
| 172.19.15.8 | 14 |
| 172.19.15.9 | 14 |
| 172.19.15.10 | 14 |
| 172.19.15.11 | 14 |
| 172.19.15.2 | 10 |
| 172.19.15.12 | 10 |
| 172.15.19.0 | 7 |
| 172.15.19.1 | 7 |
| 172.15.19.2 | 7 |
| 172.15.19.3 | 7 |
| 172.15.19.4 | 7 |
| 172.15.19.5 | 7 |
| 172.15.19.6 | 7 |

### Distribuição por sub-rede /24

| Sub-rede | SYNs |
|---|---:|
| 172.15.19.0/24 | 1.792 |
| 172.19.15.0/24 | 120 |


Distribuição dos SYNs por sub-rede

- 172.15.19.0/24 - 1.792 · 93,7%
- 172.19.15.0/24 - 120 · 6,3%

Percentuais calculados sobre os 1.912 SYNs reportados.

O predomínio de 172.15.19.0/24 sugere que a maior parte das tentativas está direcionada a esse intervalo, enquanto uma parcela menor atinge 172.19.15.0/24.

Isso é compatível com varredura SMB. Entretanto, há uma inconsistência que precisamos resolver: a análise anterior identificou 289 tuplas de SYN distintas, mas agora temos 1.792 SYNs em uma única sub-rede. Os resultados podem ser válidos, mas precisamos verificar como os contadores foram construídos e se estamos contando somente o IP de origem esperado.


---

## 11 - Construir as características por IP

#### Vamos calcular, para cada IP de origem:

- pkt_count: quantidade de pacotes.
- byte_count: volume de bytes.
- unique_dst: quantidade de destinos distintos.
- pkt_count_on_target: pacotes direcionados à porta TCP/445.
- byte_count_on_target: bytes direcionados à porta TCP/445.
- unique_dst_on_target: destinos distintos na porta TCP/445.
- syn_count: quantidade de SYNs iniciais.
- rst_count: quantidade de pacotes com flag RST.
- duration_s: duração observada da atividade.
- pps: média de pacotes por segundo.
- pps_target: média de pacotes por segundo envolvendo a porta 445.
- fraction_to_target: proporção de pacotes direcionados à porta 445.

##### Vamos usar o Scapy para ler a captura e o pandas para organizar os dados. 

#### O que observar na saída

1. A tabela deve conter vários IPs, e não apenas 172.19.15.200.
2. 172.19.15.200 deve aparecer entre os IPs com maior volume direcionado à porta 445.
3. unique_dst_on_target deverá mostrar a diversidade de destinos por origem.
4. syn_count conta SYNs observados, incluindo eventuais retransmissões; não equivale ao número de conexões únicas.

- Uma observação metodológica: neste primeiro cálculo, pkt_count_on_target considera pacotes cujo destino é a porta 445, e não respostas cuja porta de origem é 445. Isso é intencional para destacar tentativas de conexão e manter a métrica coerente com a investigação.

In [11]:

from collections import defaultdict
import pandas as pd
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP, UDP

# Contadores e conjuntos por IP de origem
stats = defaultdict(lambda: {
    "pkt_count": 0,
    "byte_count": 0,
    "destinos": set(),
    "pkt_count_on_target": 0,
    "byte_count_on_target": 0,
    "destinos_on_target": set(),
    "syn_count": 0,
    "rst_count": 0,
    "first_ts": None,
    "last_ts": None,
})

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt:
            continue

        ip_src = pkt[IP].src
        ip_dst = pkt[IP].dst
        ts = float(pkt.time)
        tamanho = len(pkt)

        s = stats[ip_src]

        s["pkt_count"] += 1
        s["byte_count"] += tamanho
        s["destinos"].add(ip_dst)

        if s["first_ts"] is None or ts < s["first_ts"]:
            s["first_ts"] = ts

        if s["last_ts"] is None or ts > s["last_ts"]:
            s["last_ts"] = ts

        # Identificar portas TCP/UDP
        porta_src = None
        porta_dst = None

        if TCP in pkt:
            tcp = pkt[TCP]
            porta_src = int(tcp.sport)
            porta_dst = int(tcp.dport)

            flags = int(tcp.flags)

            # SYN inicial: SYN presente e ACK ausente
            if (flags & 0x02) and not (flags & 0x10):
                s["syn_count"] += 1

            if flags & 0x04:
                s["rst_count"] += 1

        elif UDP in pkt:
            porta_src = int(pkt[UDP].sport)
            porta_dst = int(pkt[UDP].dport)

        # Consideramos "alvo" quando o destino é a porta 445
        if porta_dst == 445:
            s["pkt_count_on_target"] += 1
            s["byte_count_on_target"] += tamanho
            s["destinos_on_target"].add(ip_dst)

# Transformar os contadores em DataFrame
rows = []

for ip, s in stats.items():
    duracao = s["last_ts"] - s["first_ts"]

    rows.append({
        "ip": ip,
        "pkt_count": s["pkt_count"],
        "byte_count": s["byte_count"],
        "unique_dst": len(s["destinos"]),
        "pkt_count_on_target": s["pkt_count_on_target"],
        "byte_count_on_target": s["byte_count_on_target"],
        "unique_dst_on_target": len(s["destinos_on_target"]),
        "syn_count": s["syn_count"],
        "rst_count": s["rst_count"],
        "first_ts": s["first_ts"],
        "last_ts": s["last_ts"],
        "duration_s": duracao,
        "pps": s["pkt_count"] / duracao if duracao > 0 else 0,
        "pps_target": (
            s["pkt_count_on_target"] / duracao if duracao > 0 else 0
        ),
        "fraction_to_target": (
            s["pkt_count_on_target"] / s["pkt_count"]
            if s["pkt_count"] > 0 else 0
        ),
    })

df_features = pd.DataFrame(rows)

print("Dimensões da tabela:", df_features.shape)
print("Quantidade de IPs:", df_features["ip"].nunique())
print("\nCaracterísticas calculadas:")
print(df_features.columns.tolist())

print("\nTop 10 IPs por pacotes direcionados à porta 445:")
print(
    df_features.sort_values(
        "pkt_count_on_target", ascending=False
    )[["ip", "pkt_count", "pkt_count_on_target",
       "unique_dst_on_target", "syn_count", "rst_count"]]
    .head(10)
    .to_string(index=False)
)


Dimensões da tabela: (79, 15)
Quantidade de IPs: 79

Características calculadas:
['ip', 'pkt_count', 'byte_count', 'unique_dst', 'pkt_count_on_target', 'byte_count_on_target', 'unique_dst_on_target', 'syn_count', 'rst_count', 'first_ts', 'last_ts', 'duration_s', 'pps', 'pps_target', 'fraction_to_target']

Top 10 IPs por pacotes direcionados à porta 445:
             ip  pkt_count  pkt_count_on_target  unique_dst_on_target  syn_count  rst_count
  172.19.15.200      30219                 1920                   269       2396          4
    172.19.15.1       6555                    0                     0         23          0
  172.19.15.254       8980                    0                     0          0          0
    96.45.45.45        147                    0                     0          0          0
   20.106.86.13         31                    0                     0          0          0
 185.125.190.18          8                    0                     0          0          0


Resultado 

- Dimensões da tabela: (79, 15)
- Quantidade de IPs: 79


Top 10 IPs por pacotes direcionados à porta 445:

| ip             | pkt_count | pkt_count_on_target | unique_dst_on_target | syn_count | rst_count |
|----------------|----------:|--------------------:|---------------------:|----------:|----------:|
| 172.19.15.200  | 30219     | 1920                | 269                  | 2396      | 4         |
| 172.19.15.1    | 6555      | 0                   | 0                    | 23        | 0         |
| 172.19.15.254  | 8980      | 0                   | 0                    | 0         | 0         |
| 96.45.45.45    | 147       | 0                   | 0                    | 0         | 0         |
| 20.106.86.13   | 31        | 0                   | 0                    | 0         | 0         |
| 185.125.190.18 | 8         | 0                   | 0                    | 0         | 0         |
| 172.19.15.6    | 19204     | 0                   | 0                    | 128       | 35        |
| 172.172.255.218| 50        | 0                   | 0                    | 0         | 0         |
| 20.22.113.133  | 46        | 0                   | 0                    | 0         | 0         |
| 104.208.16.95  | 12        | 0                   | 0                    | 0         | 0         |


#### Validação do resultado

|Indicador	                          |  Resultado do 172.19.15.200|
|------------------------------------:|---------------------------:|
|Pacotes totais	                      |                      30.219|
|Bytes totais	                      |                          — |
|Pacotes destinados à porta 445	      |                       1.920|
|Destinos únicos na porta 445	      |                         269|
|SYNs iniciais observados	          |                       2.396|
|Pacotes RST	                      |                           4|


- Atenção: o valor de syn_count é maior que os 1.912 SYNs destinados à porta 445 porque essa característica conta SYNs em qualquer porta TCP, não apenas na 445. Isso está correto para a métrica geral, mas precisamos ter essa diferença clara na interpretação.

---

### 12 - Preparando o Isolation Forest

Vamos aplicar o algoritmo IsolationForest, que identifica registros com características diferentes do comportamento geral. Ele não precisa de rótulos prévios de IPs infectados.

Antes disso, vamos selecionar as métricas numéricas e verificar se existem valores ausentes ou infinitos. Não incluiremos ip, first_ts e last_ts como variáveis de treinamento: o endereço é um identificador, e os timestamps absolutos podem enviesar a análise.

### O que faremos depois?

Se a validação estiver limpa, vamos:

1. Tratar eventuais valores ausentes.
2. Padronizar as características com StandardScaler.
3. Treinar o IsolationForest.
4. Ordenar os 10 IPs mais anômalos.
5. Comparar a classificação do modelo com os indicadores de tráfego SMB já encontrados.

O resultado será uma lista de suspeitos baseada em anomalias estatísticas, não uma confirmação automática de infecção. Envie a saída da validação para continuarmos.

In [12]:

import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Características usadas no modelo
colunas_modelo = [
    "pkt_count",
    "byte_count",
    "unique_dst",
    "pkt_count_on_target",
    "byte_count_on_target",
    "unique_dst_on_target",
    "syn_count",
    "rst_count",
    "duration_s",
    "pps",
    "pps_target",
    "fraction_to_target",
]

X = df_features[colunas_modelo].replace(
    [np.inf, -np.inf], np.nan
)

print("Valores ausentes por característica:")
print(X.isna().sum())

print("\nValores infinitos:", np.isinf(X.to_numpy()).sum())

print("\nResumo estatístico:")
print(X.describe().round(2).to_string())


Valores ausentes por característica:
pkt_count               0
byte_count              0
unique_dst              0
pkt_count_on_target     0
byte_count_on_target    0
unique_dst_on_target    0
syn_count               0
rst_count               0
duration_s              0
pps                     0
pps_target              0
fraction_to_target      0
dtype: int64

Valores infinitos: 0

Resumo estatístico:
       pkt_count    byte_count  unique_dst  pkt_count_on_target  byte_count_on_target  unique_dst_on_target  syn_count  rst_count  duration_s     pps  pps_target  fraction_to_target
count      79.00  7.900000e+01       79.00                79.00                 79.00                 79.00      79.00      79.00       79.00   79.00       79.00               79.00
mean     2527.78  1.732952e+06        8.57                24.30               1795.44                  3.41      32.48       0.77     1707.50   16.05        0.00                0.00
std     10178.69  1.170834e+07       57.69       

Resultado 

| Característica         | Valores ausentes |
|------------------------|-----------------:|
| pkt_count              | 0 |
| byte_count             | 0 |
| unique_dst             | 0 |
| pkt_count_on_target    | 0 |
| byte_count_on_target   | 0 |
| unique_dst_on_target   | 0 |
| syn_count              | 0 |
| rst_count              | 0 |
| duration_s             | 0 |
| pps                    | 0 |
| pps_target             | 0 |
| fraction_to_target     | 0 |

**Valores infinitos:** 0

Resumo estatístico:

| Estatística | pkt_count | byte_count | unique_dst | pkt_count_on_target | byte_count_on_target | unique_dst_on_target | syn_count | rst_count | duration_s | pps | pps_target | fraction_to_target |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| count | 79.00 | 7.900000e+01 | 79.00 | 79.00 | 79.00 | 79.00 | 79.00 | 79.00 | 79.00 | 79.00 | 79.00 | 79.00 |
| mean | 2527.78 | 1.732952e+06 | 8.57 | 24.30 | 1795.44 | 3.41 | 32.48 | 0.77 | 1707.50 | 16.05 | 0.00 | 0.00 |
| std | 10178.69 | 1.170834e+07 | 57.69 | 216.02 | 15958.25 | 30.26 | 269.73 | 4.08 | 2570.05 | 57.97 | 0.04 | 0.01 |
| min | 3.00 | 1.620000e+02 | 1.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.02 | 0.00 | 0.00 | 0.00 |
| 25% | 12.00 | 1.185000e+03 | 1.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 1.44 | 0.03 | 0.00 | 0.00 |
| 50% | 19.00 | 7.684000e+03 | 1.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 60.06 | 1.00 | 0.00 | 0.00 |
| 75% | 49.00 | 2.772700e+04 | 1.00 | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 | 3750.33 | 10.82 | 0.00 | 0.00 |
| max | 69291.00 | 1.033024e+08 | 513.00 | 1920.00 | 141840.00 | 269.00 | 2396.00 | 35.00 | 6777.49 | 470.44 | 0.32 | 0.06 |


- A mediana de unique_dst é 1, enquanto o máximo é 513. Isso mostra uma grande diferença na quantidade de destinos contatados por diferentes IPs.

- O máximo de syn_count é 2.396, muito acima da mediana de zero. Esse comportamento torna essa característica especialmente relevante para investigar possíveis varreduras.

- Os valores de pps_target aparecem como 0.00 na maior parte do resumo porque foram arredondados para duas casas decimais; isso não significa necessariamente que sejam exatamente zero.

---

### 13 - Treinar o IsolationForest

O modelo vai procurar combinações de características que destoam do comportamento geral. Ele não sabe, por si só, o que é um worm: identifica anomalias estatísticas que depois precisamos interpretar com evidências da rede.

### Como interpretar o resultado

- anomaly_score: quanto menor o valor, mais anômalo o IP segundo o modelo.
- anomaly_label = -1: classificado pelo modelo como anomalia.
- anomaly_label = 1: não classificado como anomalia.
- unique_dst_on_target: ajuda a identificar origens que contataram muitos destinos pela porta 445.

O parâmetro contamination="auto" deixa o modelo definir seu limiar de classificação; portanto, não estamos impondo previamente uma porcentagem de IPs infectados.

In [13]:

import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Preparar e padronizar as características
X = df_features[colunas_modelo].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Treinar o modelo
modelo = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42
)

modelo.fit(X_scaled)

# Adicionar os resultados à tabela
df_features["anomaly_score"] = modelo.score_samples(X_scaled)
df_features["anomaly_label"] = modelo.predict(X_scaled)

# -1 = anomalia; 1 = comportamento não classificado como anômalo
df_features["is_anomaly"] = df_features["anomaly_label"] == -1

# Menor score = mais anômalo
top10 = df_features.sort_values(
    "anomaly_score", ascending=True
).head(10)

colunas_saida = [
    "ip",
    "anomaly_score",
    "anomaly_label",
    "pkt_count",
    "unique_dst",
    "pkt_count_on_target",
    "unique_dst_on_target",
    "syn_count",
    "rst_count",
]

print("Total de IPs analisados:", len(df_features))
print("IPs classificados como anomalias:", int(df_features["is_anomaly"].sum()))

print("\nTop 10 IPs mais anômalos:")
print(top10[colunas_saida].to_string(index=False))


Total de IPs analisados: 79
IPs classificados como anomalias: 8

Top 10 IPs mais anômalos:
           ip  anomaly_score  anomaly_label  pkt_count  unique_dst  pkt_count_on_target  unique_dst_on_target  syn_count  rst_count
172.19.15.200      -0.868919             -1      30219         513                 1920                   269       2396          4
 200.25.59.49      -0.751395             -1      69291           1                    0                     0          0          0
  172.19.15.6      -0.742853             -1      19204          40                    0                     0        128         35
 192.168.15.6      -0.607963             -1      47461           1                    0                     0          1          0
  172.19.15.7      -0.603356             -1       6638          14                    0                     0         16          8
  172.19.15.1      -0.587027             -1       6555          15                    0                     0        

Resultado

Total de IPs analisados: 79
IPs classificados como anomalias: 8

Top 10 IPs mais anômalos

| ip | anomaly_score | anomaly_label | pkt_count | unique_dst | pkt_count_on_target | unique_dst_on_target | syn_count | rst_count |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| 172.19.15.200 | -0.868919 | -1 | 30219 | 513 | 1920 | 269 | 2396 | 4 |
| 200.25.59.49 | -0.751395 | -1 | 69291 | 1 | 0 | 0 | 0 | 0 |
| 172.19.15.6 | -0.742853 | -1 | 19204 | 40 | 0 | 0 | 128 | 35 |
| 192.168.15.6 | -0.607963 | -1 | 47461 | 1 | 0 | 0 | 1 | 0 |
| 172.19.15.7 | -0.603356 | -1 | 6638 | 14 | 0 | 0 | 16 | 8 |
| 172.19.15.1 | -0.587027 | -1 | 6555 | 15 | 0 | 0 | 23 | 0 |
| 2.17.213.43 | -0.564830 | -1 | 9077 | 1 | 0 | 0 | 0 | 0 |
| 172.19.15.254 | -0.559638 | -1 | 8980 | 10 | 0 | 0 | 0 | 0 |
| 172.19.15.2 | -0.488485 | 1 | 58 | 6 | 0 | 0 | 2 | 2 |
| 72.153.5.134 | -0.441974 | 1 | 130 | 1 | 0 | 0 | 0 | 4 |


### Principal IP para investigação

- 172.19.15.200
- Score: -0,868919 · Classificado como anomalia

### O que os outros IPs podem indicar

| IP | Evidência observada | Interpretação inicial |
|---|---|---|
| `172.19.15.200` | 513 destinos únicos, 1.920 pacotes destinados à porta 445 | Prioridade máxima para investigar possível varredura SMB |
| `200.25.59.49` | 69.291 pacotes, apenas 1 destino único | Volume muito elevado, mas pode representar tráfego concentrado; investigar protocolo e direção |
| `172.19.15.6` | 19.204 pacotes, 35 RSTs e 40 destinos | Pode ser tráfego de serviço, conexões recusadas ou atividade incomum |
| `192.168.15.6` | 47.461 pacotes, apenas 1 destino único | Alto volume concentrado; precisa de contexto antes de classificar como malicioso |
| `172.19.15.7` | 6.638 pacotes, 14 destinos e 8 RSTs | Vale verificar a cronologia e as portas envolvidas |
| `172.19.15.1` | 6.555 pacotes e 15 destinos | Pode ser infraestrutura de rede; confirmar a função do endereço |
| `2.17.213.43` | 9.077 pacotes, apenas 1 destino único | Investigar o tipo de tráfego e se é comunicação externa legítima |
| `172.19.15.254` | 8.980 pacotes e 10 destinos | Pode ser infraestrutura; validar o papel do host |



Importante: anomalia não é sinônimo de malware. O Isolation Forest identifica comportamentos estatisticamente diferentes dos demais. Um servidor, gateway ou serviço legítimo também pode ser classificado como anômalo.

Além disso, o resultado mostra 172.19.15.200 com unique_dst = 513, enquanto a análise específica da porta 445 encontrou 269 destinos. Isso é coerente: o primeiro valor considera todos os destinos de pacotes originados pelo IP; o segundo considera somente os pacotes destinados à porta 445.


---

### 14 - investigar a cronologia

Agora precisamos verificar se o IP suspeito também aparece no início da atividade. Isso ajuda a investigar a hipótese de paciente zero, mas o primeiro IP a apresentar tráfego suspeito na captura não é necessariamente o primeiro infectado.

### O que vamos procurar

- Se 172.19.15.200 começou a transmitir antes dos demais IPs suspeitos.
- Se outros IPs começaram a apresentar tráfego anormal depois dele.
- Se os IPs com muitos pacotes e poucos destinos têm um perfil diferente da atividade SMB observada.
- Se a ordem temporal sustenta ou enfraquece a hipótese de propagação.

In [14]:
import pandas as pd

# Converter os timestamps Unix para data/hora UTC
df_features["first_time_utc"] = pd.to_datetime(
    df_features["first_ts"], unit="s", utc=True
)

df_features["last_time_utc"] = pd.to_datetime(
    df_features["last_ts"], unit="s", utc=True
)

# Mostrar as anomalias em ordem cronológica
colunas_tempo = [
    "ip",
    "first_time_utc",
    "last_time_utc",
    "anomaly_score",
    "pkt_count",
    "unique_dst",
    "pkt_count_on_target",
    "unique_dst_on_target",
    "syn_count",
    "rst_count",
]

anomalias_tempo = (
    df_features[df_features["is_anomaly"]]
    .sort_values("first_ts")
)

print("Anomalias ordenadas pelo primeiro pacote observado:\n")
print(
    anomalias_tempo[colunas_tempo].to_string(index=False)
)

Anomalias ordenadas pelo primeiro pacote observado:

           ip                      first_time_utc                       last_time_utc  anomaly_score  pkt_count  unique_dst  pkt_count_on_target  unique_dst_on_target  syn_count  rst_count
172.19.15.254 2025-11-02 19:07:03.231810093+00:00 2025-11-02 21:00:00.725939035+00:00      -0.559638       8980          10                    0                     0          0          0
  172.19.15.1 2025-11-02 19:07:03.232404947+00:00 2025-11-02 21:00:00.726562977+00:00      -0.587027       6555          15                    0                     0         23          0
  172.19.15.6 2025-11-02 19:07:08.413656950+00:00 2025-11-02 21:00:00.193089962+00:00      -0.742853      19204          40                    0                     0        128         35
  172.19.15.7 2025-11-02 19:07:20.528958082+00:00 2025-11-02 20:58:17.029124022+00:00      -0.603356       6638          14                    0                     0         16          8
 2

Resultado

#### Anomalias ordenadas pelo primeiro pacote observado

| ip | first_time_utc | last_time_utc | anomaly_score | pkt_count | unique_dst | pkt_count_on_target | unique_dst_on_target | syn_count | rst_count |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|
| 172.19.15.254 | 2025-11-02 19:07:03.231810093+00:00 | 2025-11-02 21:00:00.725939035+00:00 | -0.559638 | 8980 | 10 | 0 | 0 | 0 | 0 |
| 172.19.15.1 | 2025-11-02 19:07:03.232404947+00:00 | 2025-11-02 21:00:00.726562977+00:00 | -0.587027 | 6555 | 15 | 0 | 0 | 23 | 0 |
| 172.19.15.6 | 2025-11-02 19:07:08.413656950+00:00 | 2025-11-02 21:00:00.193089962+00:00 | -0.742853 | 19204 | 40 | 0 | 0 | 128 | 35 |
| 172.19.15.7 | 2025-11-02 19:07:20.528958082+00:00 | 2025-11-02 20:58:17.029124022+00:00 | -0.603356 | 6638 | 14 | 0 | 0 | 16 | 8 |
| 200.25.59.49 | 2025-11-02 19:07:43.749815941+00:00 | 2025-11-02 19:10:11.038598061+00:00 | -0.751395 | 69291 | 1 | 0 | 0 | 0 | 0 |
| 172.19.15.200 | 2025-11-02 19:11:37.607125044+00:00 | 2025-11-02 20:51:57.449702978+00:00 | -0.868919 | 30219 | 513 | 1920 | 269 | 2396 | 4 |
| 2.17.213.43 | 2025-11-02 19:13:13.162132978+00:00 | 2025-11-02 19:14:14.587248087+00:00 | -0.564830 | 9077 | 1 | 0 | 0 | 0 | 0 |
| 192.168.15.6 | 2025-11-02 19:17:34.386559963+00:00 | 2025-11-02 20:51:57.455390930+00:00 | -0.607963 | 47461 | 1 | 0 | 0 | 1 | 0 |


#### O que a cronologia revela

| IP | Primeiro pacote observado (UTC) | Leitura inicial |
|---|---|---|
| `172.19.15.254` | 19:07:03 | Aparece primeiro; função do host ainda precisa ser confirmada |
| `172.19.15.1` | 19:07:03 | Aparece praticamente no mesmo instante |
| `172.19.15.6` | 19:07:08 | Tráfego anterior ao do IP mais suspeito |
| `172.19.15.7` | 19:07:20 | Também aparece antes do tráfego observado do `.200` |
| `200.25.59.49` | 19:07:43 | Volume elevado concentrado em um destino |
| `172.19.15.200` | 19:11:37 | Maior pontuação de anomalia e forte atividade SMB |
| `2.17.213.43` | 19:13:13 | Tráfego concentrado em um destino |
| `192.168.15.6` | 19:17:34 | Volume elevado concentrado em um destino |


Horários em UTC; para Porto Alegre, subtraia 3 horas.

Há uma distinção essencial: first_time_utc representa o primeiro pacote de qualquer tipo observado para cada IP de origem, não o início da atividade maliciosa. Um host pode estar ativo há vários minutos antes de começar a varrer a rede.


---

### 15 - Quem iniciou as conexões SMB primeiro

Vamos separar o tráfego geral das tentativas de conexão TCP para a porta 445. Assim, conseguiremos comparar o início das tentativas SMB entre os hosts, em vez de usar o primeiro pacote de cada IP.

### Como vamos interpretar

- Primeiro SYN: identifica quem iniciou primeiro uma tentativa SMB entre as origens encontradas.
- Muitos destinos únicos: pode indicar varredura ou tentativa de propagação.
- Muitos SYNs para poucos destinos: pode indicar tentativas repetidas, retransmissões ou outro padrão de conexão.
- Ordem temporal: ajuda a investigar a sequência, mas não prova quem foi infectado primeiro.

In [15]:
from collections import defaultdict
import pandas as pd
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

smb_syn = defaultdict(lambda: {
    "first_syn": None,
    "syn_count": 0,
    "destinos": set()
})

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]

        # SYN inicial destinado à porta SMB, sem flag ACK
        if int(tcp.dport) == 445 and (int(tcp.flags) & 0x02) and not (int(tcp.flags) & 0x10):
            ip_src = pkt[IP].src
            ip_dst = pkt[IP].dst
            ts = float(pkt.time)

            s = smb_syn[ip_src]
            s["syn_count"] += 1
            s["destinos"].add(ip_dst)

            if s["first_syn"] is None or ts < s["first_syn"]:
                s["first_syn"] = ts

rows_smb = []

for ip, s in smb_syn.items():
    rows_smb.append({
        "ip": ip,
        "first_syn_utc": pd.to_datetime(
            s["first_syn"], unit="s", utc=True
        ),
        "syn_count": s["syn_count"],
        "destinos_unicos": len(s["destinos"]),
    })

df_smb_syn = pd.DataFrame(rows_smb)

print("Origens com SYN para a porta 445, em ordem cronológica:\n")
print(
    df_smb_syn.sort_values("first_syn_utc")
    .head(20)
    .to_string(index=False)
)

Origens com SYN para a porta 445, em ordem cronológica:

           ip                       first_syn_utc  syn_count  destinos_unicos
172.19.15.200 2025-11-02 19:33:31.212356091+00:00       1912              269


Resultado

Origens com SYN para a porta 445, em ordem cronológica:

| IP | first_syn_utc | syn_count | destinos_unicos |
|---|---|---:|---:|
| 172.19.15.200 | 2025-11-02 19:33:31.212356091+00:00 | 1912 | 269 |

----

### 16 - investigar a distribuição temporal dos SYNs

Vamos verificar se a atividade ocorreu de forma contínua ou em rajadas. Isso pode ajudar a caracterizar o padrão de varredura.

In [16]:
from collections import Counter
import pandas as pd
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

syn_por_minuto = Counter()

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]

        if (
            int(tcp.dport) == 445
            and (int(tcp.flags) & 0x02)
            and not (int(tcp.flags) & 0x10)
            and pkt[IP].src == "172.19.15.200"
        ):
            minuto = int(float(pkt.time) // 60)
            syn_por_minuto[minuto] += 1

df_smb_tempo = pd.DataFrame([
    {
        "minuto_utc": pd.to_datetime(
            minuto * 60, unit="s", utc=True
        ),
        "syn_count": quantidade
    }
    for minuto, quantidade in sorted(syn_por_minuto.items())
])

print("Minutos com mais SYNs para a porta 445:\n")
print(
    df_smb_tempo.sort_values(
        "syn_count", ascending=False
    ).head(15).to_string(index=False)
)

print("\nResumo:")
print("Minutos com atividade:", len(df_smb_tempo))
print("Maior quantidade em um minuto:", df_smb_tempo["syn_count"].max())
print("Média por minuto ativo:", round(df_smb_tempo["syn_count"].mean(), 2))

Minutos com mais SYNs para a porta 445:

               minuto_utc  syn_count
2025-11-02 19:33:00+00:00        110
2025-11-02 19:38:00+00:00         42
2025-11-02 19:39:00+00:00         42
2025-11-02 19:40:00+00:00         42
2025-11-02 19:41:00+00:00         42
2025-11-02 19:43:00+00:00         42
2025-11-02 19:42:00+00:00         42
2025-11-02 19:44:00+00:00         42
2025-11-02 19:45:00+00:00         42
2025-11-02 19:49:00+00:00         42
2025-11-02 19:46:00+00:00         42
2025-11-02 19:47:00+00:00         42
2025-11-02 19:48:00+00:00         42
2025-11-02 19:51:00+00:00         42
2025-11-02 19:50:00+00:00         42

Resumo:
Minutos com atividade: 49
Maior quantidade em um minuto: 110
Média por minuto ativo: 39.02


Resultado

Minutos com mais SYNs para a porta 445

| minuto_utc | syn_count |
|---|---:|
| 2025-11-02 19:33:00+00:00 | 110 |
| 2025-11-02 19:38:00+00:00 | 42 |
| 2025-11-02 19:39:00+00:00 | 42 |
| 2025-11-02 19:40:00+00:00 | 42 |
| 2025-11-02 19:41:00+00:00 | 42 |
| 2025-11-02 19:43:00+00:00 | 42 |
| 2025-11-02 19:42:00+00:00 | 42 |
| 2025-11-02 19:44:00+00:00 | 42 |
| 2025-11-02 19:45:00+00:00 | 42 |
| 2025-11-02 19:49:00+00:00 | 42 |
| 2025-11-02 19:46:00+00:00 | 42 |
| 2025-11-02 19:47:00+00:00 | 42 |
| 2025-11-02 19:48:00+00:00 | 42 |
| 2025-11-02 19:51:00+00:00 | 42 |
| 2025-11-02 19:50:00+00:00 | 42 |


Resumo:
Minutos com atividade: 49
Maior quantidade em um minuto: 110
Média por minuto ativo: 39.02

Esses números mostram um padrão interessante: o 172.19.15.200 enviou SYNs para a porta 445 durante 49 minutos distintos, com média de aproximadamente 39 SYNs por minuto ativo e pico de 110.

O detalhe mais importante é que, após o pico inicial das 19:33 UTC, aparecem vários minutos com exatamente 42 SYNs. Isso pode indicar uma rotina repetitiva de varredura, mas precisamos verificar se são conexões para destinos diferentes ou repetições para os mesmos destinos.

---

### 17 - Padrão de destinos

Vamos agrupar os SYNs por minuto e contar quantos destinos distintos foram contatados em cada minuto. Assim poderemos diferenciar melhor uma varredura distribuída de tentativas repetidas.


### Como interpretar a saída

- 42 SYNs e perto de 42 destinos únicos: compatível com uma tentativa de contato distribuída entre vários hosts.
- 42 SYNs e poucos destinos únicos: indica repetição de SYNs para alguns destinos.
- 42 SYNs e um destino único: pode representar tentativas repetidas, retransmissões ou duplicações na captura.

In [ ]:
from collections import defaultdict
import pandas as pd
from scapy.utils import PcapNgReader
from scapy.layers.inet import IP, TCP

por_minuto = defaultdict(lambda: {
    "syn_count": 0,
    "destinos": set()
})

with PcapNgReader(str(pcap_path)) as leitor:
    for pkt in leitor:
        if IP not in pkt or TCP not in pkt:
            continue

        tcp = pkt[TCP]

        if (
            pkt[IP].src == "172.19.15.200"
            and int(tcp.dport) == 445
            and (int(tcp.flags) & 0x02)
            and not (int(tcp.flags) & 0x10)
        ):
            minuto = int(float(pkt.time) // 60)
            por_minuto[minuto]["syn_count"] += 1
            por_minuto[minuto]["destinos"].add(pkt[IP].dst)

df_smb_minuto = pd.DataFrame([
    {
        "minuto_utc": pd.to_datetime(
            minuto * 60, unit="s", utc=True
        ),
        "syn_count": dados["syn_count"],
        "destinos_unicos": len(dados["destinos"]),
        "syns_por_destino": round(
            dados["syn_count"] / len(dados["destinos"]), 2
        ) if dados["destinos"] else 0
    }
    for minuto, dados in sorted(por_minuto.items())
])

print("Minutos com mais SYNs e destinos distintos:\n")
print(
    df_smb_minuto.sort_values(
        "syn_count", ascending=False
    ).head(15).to_string(index=False)
)

print("\nResumo de destinos únicos por minuto:")
print(df_smb_minuto["destinos_unicos"].describe().round(2))

Resultado

Minutos com mais SYNs e destinos distintos:

| minuto_utc | syn_count | destinos_unicos | syns_por_destino |
|---|---:|---:|---:|
| 2025-11-02 19:33:00+00:00 | 110 | 12 | 9.17 |
| 2025-11-02 19:38:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:39:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:40:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:41:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:43:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:42:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:44:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:45:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:49:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:46:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:47:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:48:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:51:00+00:00 | 42 | 7 | 6.00 |
| 2025-11-02 19:50:00+00:00 | 42 | 7 | 6.00 |


| Estatística | destinos_unicos |
|---|---:|
| count | 49.00 |
| mean | 6.43 |
| std | 2.01 |
| min | 1.00 |
| 25% | 7.00 |
| 50% | 7.00 |
| 75% | 7.00 |
| max | 12.00 |

---

### 18 - Concluir a análise com z-scores

Identificar quais características do 172.19.15.200 mais se afastam do padrão dos demais IPs.

In [ ]:
import pandas as pd

colunas_modelo = [
    "pkt_count", "byte_count", "unique_dst",
    "pkt_count_on_target", "byte_count_on_target",
    "unique_dst_on_target", "syn_count", "rst_count",
    "duration_s", "pps", "pps_target", "fraction_to_target"
]

X = df_features[colunas_modelo]

desvio = X.std(ddof=0).replace(0, float("nan"))
z_scores = (X - X.mean()) / desvio

ip_alvo = "172.19.15.200"

resultado = pd.DataFrame({
    "caracteristica": colunas_modelo,
    "valor_ip": df_features.loc[
        df_features["ip"] == ip_alvo, colunas_modelo
    ].iloc[0].values,
    "z_score": z_scores.loc[
        df_features["ip"] == ip_alvo
    ].iloc[0].values
})

resultado["z_score_abs"] = resultado["z_score"].abs()
resultado = resultado.sort_values("z_score_abs", ascending=False)

print("Características mais fora do padrão para", ip_alvo)
print(resultado.round(3).to_string(index=False))

Resultado

**Características mais fora do padrão para `172.19.15.200`:**

| Característica | Valor do IP | z_score | z_score_abs |
|---|---:|---:|---:|
| byte_count_on_target | 141840.000 | 8.832 | 8.832 |
| pps_target | 0.319 | 8.832 | 8.832 |
| pkt_count_on_target | 1920.000 | 8.832 | 8.832 |
| fraction_to_target | 0.064 | 8.832 | 8.832 |
| unique_dst_on_target | 269.000 | 8.832 | 8.832 |
| syn_count | 2396.000 | 8.819 | 8.819 |
| unique_dst | 513.000 | 8.799 | 8.799 |
| pkt_count | 30219.000 | 2.738 | 2.738 |
| duration_s | 6019.843 | 1.689 | 1.689 |
| rst_count | 4.000 | 0.796 | 0.796 |
| pps | 5.020 | -0.192 | 0.192 |
| byte_count | 3274192.000 | 0.132 | 0.132 |

### Interpretação dos z-scores

O 172.19.15.200 apresenta desvios muito elevados nas características relacionadas à porta 445 e à diversidade de destinos.

| Característica | Z-score | Interpretação |
|---|---:|---|
| Pacotes destinados à porta 445 | +8,832 | Extremamente acima da média |
| Destinos únicos na porta 445 | +8,832 | Comportamento muito fora do padrão |
| SYNs TCP | +8,819 | Quantidade excepcionalmente elevada |
| Destinos únicos em geral | +8,799 | Contato com muitos hosts |
| Total de pacotes | +2,738 | Volume acima da média |
| Pacotes por segundo (`pps`) | −0,192 | Taxa global próxima da média |


A principal conclusão: não é apenas o volume de tráfego que diferencia esse host. É a combinação de muitas conexões iniciadas, muitos destinos e forte concentração da atividade em SMB.

Uma observação metodológica: os cinco z-scores de +8,832 são iguais porque essas características apresentam valores zero para praticamente todos os outros IPs, gerando a mesma média e o mesmo desvio-padrão. Não devem ser interpretados como cinco evidências estatisticamente independentes.


## Conclusão do laboratório

`Veredito técnico`

Principal host atacante suspeito

**IP:** `172.19.15.200`

A captura apresenta evidências consistentes com atividade automatizada de varredura SMB: **1.912 SYNs** destinados à porta TCP 445, **269 destinos distintos** e características estatísticas muito discrepantes em relação aos demais IPs. O Isolation Forest também classificou esse endereço como a anomalia mais significativa entre os 79 IPs analisados.

A hipótese de propagação de worm é plausível, mas a confirmação de infecção, exploração bem-sucedida e identificação do paciente zero exigiria evidências adicionais.

